In [17]:
from pathlib import Path
import pandas as pd

# =========================
# 1. CARGA DE PARQUETS
# =========================
path = Path(r"..\..\Raw\EV_prediction_raw_data")
files = list(path.glob("*.parquet"))

dfs = []

for f in files:
    df_temp = pd.read_parquet(f)
    
    # fecha del dataset (opcional, para control)
    year, month = f.stem.split("_")
    df_temp["fecha_dataset"] = pd.to_datetime(f"{year}-{month}-01")
    
    dfs.append(df_temp)

df_ev_raw = pd.concat(dfs, ignore_index=True)

print("Dimensión inicial:", df_ev_raw.shape)

Dimensión inicial: (14588498, 8)


In [18]:
# =========================
# 2. NORMALIZACIÓN DE COLUMNAS
# =========================
df_ev_raw.columns = (
    df_ev_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

In [19]:
# =========================
# 3. LIMPIEZA BÁSICA DE TEXTO
# =========================
for col in df_ev_raw.select_dtypes(include="object").columns:
    df_ev_raw[col] = (
        df_ev_raw[col]
        .astype(str)
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    )

In [20]:
# =========================
# 4. LIMPIEZA DE FECHA DE MATRICULACIÓN
# =========================

# asegurar formato correcto (DDMMYYYY)
df_ev_raw["fec_matricula"] = (
    df_ev_raw["fec_matricula"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(8)
)

# convertir a datetime
df_ev_raw["fecha_matricula"] = pd.to_datetime(
    df_ev_raw["fec_matricula"],
    format="%d%m%Y",
    errors="coerce"
)

In [21]:
# =========================
# 5. VARIABLES TEMPORALES
# =========================
df_ev_raw["year"] = df_ev_raw["fecha_matricula"].dt.year
df_ev_raw["month"] = df_ev_raw["fecha_matricula"].dt.month
df_ev_raw["year_month"] = df_ev_raw["fecha_matricula"].dt.to_period("M")

In [22]:
# =========================
# 6. FILTRO DE FECHAS VÁLIDAS
# =========================

df_ev_raw = df_ev_raw[
    df_ev_raw["fecha_matricula"].notna() &
    (df_ev_raw["year"] >= 2000) &
    (df_ev_raw["year"] <= 2026)
]

In [23]:
# =========================
# 7. LIMPIEZA DE DUPLICADOS
# =========================
df_ev_raw = df_ev_raw.drop_duplicates()

In [24]:
# =========================
# 8. LIMPIEZA COLUMNA EV
# =========================

df_ev_raw["categoría_vehículo_eléctrico"] = (
    df_ev_raw["categoría_vehículo_eléctrico"]
    .astype(str)
    .str.strip()
    .replace({"": pd.NA, "nan": pd.NA})
)

In [25]:
# =========================
# 9. FILTRAR SOLO VEHÍCULOS ELÉCTRICOS
# =========================

df_ev = df_ev_raw[
    df_ev_raw["categoría_vehículo_eléctrico"].notna()
].copy()

In [26]:
# =========================
# 10. DATASET FINAL LIMPIO
# =========================

print("Dimensión EV:", df_ev.shape)

df_ev.head()

Dimensión EV: (3359919, 12)


,fec_matricula,marca_itv,modelo_itv,cod_tipo,cod_propulsion_itv,clave_tramite,categoría_vehículo_eléctrico,fecha_dataset,fecha_matricula,year,month,year_month
0,02012015,VOLKSWAGEN,TIGUAN,40,1,1,<NA>,2015-01-01,2015-01-02,2015.0,1.0,2015-01
2,02012015,LEXUS,LEXUS GS300H,40,0,1,HEV,2015-01-01,2015-01-02,2015.0,1.0,2015-01
3,02012015,BMW,X3 XDRIVE20D,40,1,1,<NA>,2015-01-01,2015-01-02,2015.0,1.0,2015-01
4,02012015,ROS-ROCA,160E,0D,1,1,<NA>,2015-01-01,2015-01-02,2015.0,1.0,2015-01
5,02012015,AUDI,A1 SPORTBACK,40,1,1,<NA>,2015-01-01,2015-01-02,2015.0,1.0,2015-01
